In [15]:
## Collect Positives and Negatives, with Free Zone Augmentation

import os
import shutil
from PIL import Image
import numpy as np
import random

source_root = '../data/sorted'
target_dir_pos = '../classification/positive'
target_dir_neg = '../classification/negative'

# Augmentation settings for free_zone
AUGMENT_FREE_ZONE = True
AUG_PER_IMAGE = 3  # Number of augmented images per original

def augment_image(img):
    """Return a list of augmented PIL Images from the input image."""
    augmented = []
    # 1. Horizontal flip
    augmented.append(img.transpose(Image.FLIP_LEFT_RIGHT))
    # 2. Random rotation
    angle = random.choice([90, 180, 270])
    augmented.append(img.rotate(angle))
    # 3. Color jitter (simple brightness change)
    factor = random.uniform(0.7, 1.3)
    augmented.append(Image.fromarray(np.clip(np.array(img) * factor, 0, 255).astype(np.uint8)))
    return augmented

# Create the target directories if they don't exist
os.makedirs(target_dir_pos, exist_ok=True)
os.makedirs(target_dir_neg, exist_ok=True)

for folder in os.listdir(source_root):
    folder_path = os.path.join(source_root, folder)
    if os.path.isdir(folder_path) and folder != 'free_zone':
        for fname in os.listdir(folder_path):
            src_file = os.path.join(folder_path, fname)
            if os.path.isfile(src_file):
                dst_file = os.path.join(target_dir_pos, fname)
                # If file with same name exists, rename to avoid overwrite
                base, ext = os.path.splitext(fname)
                counter = 1
                while os.path.exists(dst_file):
                    dst_file = os.path.join(target_dir_pos, f"{base}_{counter}{ext}")
                    counter += 1
                shutil.copy2(src_file, dst_file)
    elif os.path.isdir(folder_path) and folder == 'free_zone':
        for fname in os.listdir(folder_path):
            src_file = os.path.join(folder_path, fname)
            if os.path.isfile(src_file):
                # Copy original
                dst_file = os.path.join(target_dir_neg, fname)
                base, ext = os.path.splitext(fname)
                counter = 1
                while os.path.exists(dst_file):
                    dst_file = os.path.join(target_dir_neg, f"{base}_{counter}{ext}")
                    counter += 1
                shutil.copy2(src_file, dst_file)
                # Augment and save more data for free_zone
                if AUGMENT_FREE_ZONE:
                    try:
                        img = Image.open(src_file)
                        aug_imgs = augment_image(img)
                        for i, aug_img in enumerate(aug_imgs):
                            aug_fname = f"{base}_aug{i+1}{ext}"
                            aug_dst_file = os.path.join(target_dir_neg, aug_fname)
                            aug_counter = 1
                            while os.path.exists(aug_dst_file):
                                aug_fname = f"{base}_aug{i+1}_{aug_counter}{ext}"
                                aug_dst_file = os.path.join(target_dir_neg, aug_fname)
                                aug_counter += 1
                            aug_img.save(aug_dst_file)
                    except Exception as e:
                        print(f"Augmentation failed for {src_file}: {e}")

In [18]:
from sklearn.model_selection import train_test_split

def split_and_copy_sklearn_train_val_test(src_dir, dst_root, class_name, split_ratios):
    """
    Splits files in src_dir into train/val/test folders in dst_root/class_name
    according to split_ratios (dict with keys 'train', 'val', 'test').
    """
    files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    train_size = split_ratios['train']
    val_size = split_ratios['val']
    test_size = split_ratios['test']

    # First split off train
    train_files, temp_files = train_test_split(
        files, train_size=train_size, shuffle=True, random_state=42
    )
    # Then split temp into val and test
    # The proportion of val and test in the temp set
    temp_total = val_size + test_size
    val_prop = val_size / temp_total
    val_files, test_files = train_test_split(
        temp_files, train_size=val_prop, shuffle=True, random_state=42
    )

    splits = {
        'train': train_files,
        'val': val_files,
        'test': test_files
    }

    for split, split_files in splits.items():
        split_dir = os.path.join(dst_root, split, class_name)
        os.makedirs(split_dir, exist_ok=True)
        for fname in split_files:
            src_file = os.path.join(src_dir, fname)
            dst_file = os.path.join(split_dir, fname)
            base, ext = os.path.splitext(fname)
            counter = 1
            while os.path.exists(dst_file):
                dst_file = os.path.join(split_dir, f"{base}_{counter}{ext}")
                counter += 1
            shutil.copy2(src_file, dst_file)

# Define source and destination directories
src_pos = '../classification/positive'
src_neg = '../classification/negative'
dst_data = '../classification/data'

# Remove old data splits if they exist
for split in ['train', 'val', 'test']:
    for cls in ['positive', 'negative']:
        split_dir = os.path.join(dst_data, split, cls)
        if os.path.exists(split_dir):
            shutil.rmtree(split_dir)

# Split and copy with correct ratios using sklearn
split_ratios = {'train': 0.6, 'test': 0.3, 'val': 0.1}
split_and_copy_sklearn_train_val_test(src_pos, dst_data, 'positive', split_ratios)
split_and_copy_sklearn_train_val_test(src_neg, dst_data, 'negative', split_ratios)


In [34]:
import os

base = '../classification/'
for split in ['train','val']:
    print(f"\n{split.upper()}:")
    for cls in ['positive','negative']:
        path = os.path.join(base, split, cls)
        cnt  = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.png'))])
        print(f"  {cls:>8}: {cnt} images")



TRAIN:
  positive: 609 images
  negative: 300 images

VAL:
  positive: 261 images
  negative: 33 images


In [ ]:
from ultralytics import YOLO
import os
import pathlib

print("CWD =", os.getcwd())

# locate the folder where this script lives
BASE_DIR = pathlib.Path(__file__).parent

# build the path to data.yaml
DATA_PATH = BASE_DIR / "data.yaml"
# Load a model
model = YOLO("yolo11n-cls.pt")  

results = model.train(
    data=DATA_PATH,
    epochs=5,
    imgsz=64,
    batch=32,
    lr0=0.01,
    save = True
)

metrics = model.val()

CWD = /Users/igor/Desktop/thermal-landmine-detection/classification


NameError: name '__file__' is not defined

In [32]:
import os
import shutil
from PIL import Image
import numpy as np
import random
from sklearn.model_selection import train_test_split

source_root = '../data/sorted'
dst_data = '../classification/data'
os.makedirs(dst_data, exist_ok=True)


AUGMENT_FREE_ZONE = True

def augment_image(img):
    """Return a list of augmented PIL Images from the input image."""
    augmented = []
    # 1. Horizontal flip
    augmented.append(img.transpose(Image.FLIP_LEFT_RIGHT))
    # 2. Random rotation
    angle = random.choice([90, 180, 270])
    augmented.append(img.rotate(angle))
    # 3. Color jitter (simple brightness change)
    factor = random.uniform(0.7, 1.3)
    augmented.append(Image.fromarray(np.clip(np.array(img) * factor, 0, 255).astype(np.uint8)))
    return augmented

# Collect all positive and negative image file paths
positive_files = []
negative_files = []

for folder in os.listdir(source_root):
    folder_path = os.path.join(source_root, folder)
    if os.path.isdir(folder_path) and folder != 'free_zone':
        for fname in os.listdir(folder_path):
            src_file = os.path.join(folder_path, fname)
            if os.path.isfile(src_file):
                positive_files.append(src_file)
    elif os.path.isdir(folder_path) and folder == 'free_zone':
        for fname in os.listdir(folder_path):
            src_file = os.path.join(folder_path, fname)
            if os.path.isfile(src_file):
                negative_files.append(src_file)

# Shuffle the lists
random.seed(42)
random.shuffle(positive_files)
random.shuffle(negative_files)

# Split into train/val only
split_ratios = {'train': 0.7, 'val': 0.3}

def split_files(files, split_ratios):
    train_size = split_ratios['train']
    val_size = split_ratios['val']
    train_files, val_files = train_test_split(
        files, train_size=train_size, shuffle=False, random_state=None
    )
    return train_files, val_files

pos_train, pos_val = split_files(positive_files, split_ratios)
neg_train, neg_val = split_files(negative_files, split_ratios)

splits = {
    'train': {'positive': pos_train, 'negative': neg_train},
    'val':   {'positive': pos_val,   'negative': neg_val},
}

#  Remove old data splits if they exist
for split in ['train', 'val']:
    for cls in ['positive', 'negative']:
        split_dir = os.path.join(dst_data, split, cls)
        if os.path.exists(split_dir):
            shutil.rmtree(split_dir)

# Copy files to their split/class folders
for split in ['train', 'val']:
    for cls in ['positive', 'negative']:
        split_dir = os.path.join(dst_data, split, cls)
        os.makedirs(split_dir, exist_ok=True)
        for src_file in splits[split][cls]:
            fname = os.path.basename(src_file)
            base, ext = os.path.splitext(fname)
            dst_file = os.path.join(split_dir, fname)
            counter = 1
            while os.path.exists(dst_file):
                dst_file = os.path.join(split_dir, f"{base}_{counter}{ext}")
                counter += 1
            shutil.copy2(src_file, dst_file)

# Augment only the train/negative set (i.e., free_zone) after split
if AUGMENT_FREE_ZONE:
    train_neg_dir = os.path.join(dst_data, 'train', 'negative')
    neg_train_files = [f for f in os.listdir(train_neg_dir) if os.path.isfile(os.path.join(train_neg_dir, f))]
    for fname in neg_train_files:
        src_file = os.path.join(train_neg_dir, fname)
        try:
            img = Image.open(src_file)
            aug_imgs = augment_image(img)
            for i, aug_img in enumerate(aug_imgs):
                base, ext = os.path.splitext(fname)
                aug_fname = f"{base}_aug{i+1}{ext}"
                aug_dst_file = os.path.join(train_neg_dir, aug_fname)
                aug_counter = 1
                while os.path.exists(aug_dst_file):
                    aug_fname = f"{base}_aug{i+1}_{aug_counter}{ext}"
                    aug_dst_file = os.path.join(train_neg_dir, aug_fname)
                    aug_counter += 1
                aug_img.save(aug_dst_file)
        except Exception as e:
            print(f"Augmentation failed for {src_file}: {e}")
